# Lab 01 — Single-Qubit Gates and the Bloch Sphere

Apply the Pauli gates $X, Y, Z$, the Hadamard $H$, the phase gates $S, T$
and the rotations $R_x, R_y, R_z$; check their matrices, and see how each
one moves the state on the Bloch sphere.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(ROOT))

## Parameters

In [ ]:
import numpy as np

THETA = np.pi / 3   # rotation angle for Rx, Ry, Rz
AXIS = "y"          # rotation axis to explore: "x", "y" or "z"

In [ ]:
from qiskit import QuantumCircuit
from qiskit.circuit.library import XGate, YGate, ZGate, HGate, SGate, TGate
from qiskit.quantum_info import Operator, Statevector
from qiskit.visualization import plot_bloch_multivector

from common.qlab import format_state, save_circuit

if AXIS not in ("x", "y", "z"):
    raise ValueError('AXIS must be "x", "y" or "z".')

## Step 1 — Gate matrices

In [ ]:
GATES = {"X": XGate(), "Y": YGate(), "Z": ZGate(), "H": HGate(), "S": SGate(), "T": TGate()}

for name, gate in GATES.items():
    print(f"{name} =\n{np.round(Operator(gate).data, 3)}\n")

## Step 2 — Gate gallery
$Z$, $S$, $T$ only change the *phase* of $|1\rangle$, so they do nothing visible to $|0\rangle$.
Those gates are applied after an $H$, to the superposition $|+\rangle$.

In [ ]:
GALLERY = [
    ("X",    lambda qc, i: qc.x(i)),
    ("Y",    lambda qc, i: qc.y(i)),
    ("H",    lambda qc, i: qc.h(i)),
    ("H, Z", lambda qc, i: (qc.h(i), qc.z(i))),
    ("H, S", lambda qc, i: (qc.h(i), qc.s(i))),
    ("H, T", lambda qc, i: (qc.h(i), qc.t(i))),
    (f"R{AXIS}(θ)", lambda qc, i: getattr(qc, f"r{AXIS}")(THETA, i)),
]

gallery = QuantumCircuit(len(GALLERY))
for i, (_, apply) in enumerate(GALLERY):
    apply(gallery, i)

save_circuit(gallery, "circuit.png", scale=1.0);

for i, (label, apply) in enumerate(GALLERY):
    single = QuantumCircuit(1)
    apply(single, 0)
    print(f"q[{i}]  {label:<8} |0> -> {format_state(*Statevector(single).data)}")

## Step 3 — Bloch sphere of every qubit
`X`: north → south pole. `H`: onto the equator at $+x$. `Z`, `S`, `T` then rotate
about the $z$-axis by $\pi$, $\pi/2$, $\pi/4$.

In [ ]:
fig = plot_bloch_multivector(Statevector(gallery), title="Gate gallery (q0 ... q6)")
fig.savefig("bloch.png", dpi=150, bbox_inches="tight")
fig

## Verification — gate identities and expected states

In [ ]:
def same(a, b):
    """Equal as operators up to a global phase."""
    return Operator(a).equiv(Operator(b))

h = QuantumCircuit(1); h.h(0)
hzh = QuantumCircuit(1); hzh.h(0); hzh.z(0); hzh.h(0)
hxh = QuantumCircuit(1); hxh.h(0); hxh.x(0); hxh.h(0)
ss = QuantumCircuit(1); ss.s(0); ss.s(0)
tt = QuantumCircuit(1); tt.t(0); tt.t(0)
xz = QuantumCircuit(1); xz.z(0); xz.x(0)
hh = QuantumCircuit(1); hh.h(0); hh.h(0)

identities = {
    "HZH = X": same(hzh, XGate()),
    "HXH = Z": same(hxh, ZGate()),
    "S² = Z": same(ss, ZGate()),
    "T² = S": same(tt, SGate()),
    "XZ = -iY": np.allclose(Operator(xz).data, -1j * Operator(YGate()).data),
    "H² = I": same(hh, Operator(np.eye(2))),
}
for name, ok in identities.items():
    print(f"{name:<9}: {ok}")

s2 = 1 / np.sqrt(2)
c, s = np.cos(THETA / 2), np.sin(THETA / 2)
expected_rotation = {"x": [c, -1j * s], "y": [c, s], "z": [np.exp(-1j * THETA / 2), 0]}[AXIS]
expected = [
    [0, 1], [0, 1j], [s2, s2], [s2, -s2], [s2, 1j * s2], [s2, np.exp(1j * np.pi / 4) * s2],
    expected_rotation,
]

for (label, apply), vector in zip(GALLERY, expected):
    single = QuantumCircuit(1)
    apply(single, 0)
    assert Statevector(single).equiv(Statevector(vector)), label

assert all(identities.values()), identities
print("\nAll gate matrices, identities and output states verified.")

In [ ]:
gallery.draw(output="mpl", scale=1.0)